# 06 - Algoritma Karşılaştırması: Nitel (Qualitative) Analiz — LLM-as-a-Judge

**Amaç:** `Model_Base`, `Model_SFT`, `Model_RLHF` ve `Model_DPO` modellerinin
aynı test istemlerine verdiği yanıtları **körleştirerek** (blind), güçlü bir
hakem modele (GPT-4o veya Llama-3.1-70B-Instruct) sunmak ve **ikili kazanma
oranı (Win-Rate %) matrisi** çıkarmak (Zheng vd., 2023; "LLM-as-a-Judge").

Konum önyargısını (position bias) azaltmak için her karşılaştırmada A/B sırası
rastgele belirlenir.

> Bu defter, `05_Algorithm_Comparison_Quantitative.ipynb` ile aynı model
> çıktılarına (adaptörler) ihtiyaç duyar.

## 0. Ortam Kurulumu

In [ ]:
# --- Google Colab ortam kurulumu -------------------------------------------
# Bu hucre, projenin tamamini (Datasets/, src/, notebooks/, outputs/) iceren
# "Uygulama" klasorunu Google Drive'a yukledikten sonra calistirilmalidir.
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO: Kendi Google Drive yolunuza gore guncelleyin
    PROJECT_ROOT = "/content/drive/MyDrive/Uygulama"
else:
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
# NOT: Surumler, TRL'nin klasik `PPOTrainer` + `AutoModelForCausalLMWithValueHead`
# coklu-adaptor (multi-adapter) RL ozelligiyle dogrulanmis, bilinen-uyumlu bir
# kombinasyon olacak sekilde SABITLENMISTIR (API kararliligi icin).
%pip install -q -U "transformers==4.44.2" "trl==0.9.6" "peft==0.11.1" "bitsandbytes==0.43.1" "accelerate==0.32.1" "datasets==2.20.0" "numpy==1.26.4" sentencepiece
%pip install -q -U openai huggingface_hub

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import alignment_pipeline as ap

ap.set_global_seed(42)
print("Cihaz:", ap.get_device())

In [ ]:
config = ap.ExperimentConfig(
    dataset_root=os.path.join(PROJECT_ROOT, "Datasets"),
    output_root=os.path.join(PROJECT_ROOT, "outputs"),
)
ap.ensure_dir(config.output_root)
print("Taban model:", config.base_model_id)
print("Veri kok dizini:", config.dataset_root)
print("Cikti kok dizini:", config.output_root)

In [ ]:
SFT_MERGED_DIR = config.path("sft", "merged_model")
PPO_ADAPTER_DIR = config.path("ppo", "ppo_adapter")
DPO_ADAPTER_DIR = config.path("dpo", "adapter")

QUAL_DIR = config.path("qualitative")
ap.ensure_dir(QUAL_DIR)

QUANT_BITS = 4
TEST_PROMPTS = ap.DEFAULT_QUALITATIVE_TEST_PROMPTS

## 1. Modelleri Yükle ve Yanıt Üret

In [ ]:
model_paths = {
    "Model_Base": {"base_model_id": config.base_model_id, "adapter_path": None},
    "Model_SFT":  {"base_model_id": SFT_MERGED_DIR,       "adapter_path": None},
    "Model_RLHF": {"base_model_id": SFT_MERGED_DIR,       "adapter_path": PPO_ADAPTER_DIR},
    "Model_DPO":  {"base_model_id": SFT_MERGED_DIR,       "adapter_path": DPO_ADAPTER_DIR},
}

responses_by_model = {}
for name, paths in model_paths.items():
    model, tok = ap.load_model_for_inference(paths["base_model_id"], paths["adapter_path"], quant_bits=QUANT_BITS)
    responses_by_model[name] = ap.generate_batch(model, tok, TEST_PROMPTS, max_new_tokens=config.max_new_tokens)
    del model  # VRAM'i serbest bırak
    print(f"{name} için yanıtlar üretildi.")

ap.save_json(responses_by_model, os.path.join(QUAL_DIR, "responses_by_model.json"))

In [ ]:
for i, prompt in enumerate(TEST_PROMPTS[:3]):
    print("İSTEM:", prompt)
    for name, responses in responses_by_model.items():
        print(f"  [{name}] {responses[i][:200]}")
    print("-" * 100)

## 2. Hakem (Judge) Modeli Ayarları

Aşağıdaki hücrede **GPT-4o** (OpenAI API) kullanılmaktadır; alternatif olarak
Hugging Face Inference API üzerinden **Llama-3.1-70B-Instruct** kullanmak için
yorum satırındaki örneği aktif edebilirsiniz.

In [ ]:
import getpass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API anahtarınızı girin: ")

def judge_fn(prompt: str):
    return ap.judge_with_openai(prompt, model="gpt-4o", api_key=OPENAI_API_KEY)

# --- Alternatif: Llama-3.1-70B-Instruct (Hugging Face Inference API) ---
# HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token girin: ")
# def judge_fn(prompt: str):
#     return ap.judge_with_hf_inference(prompt, model="meta-llama/Llama-3.1-70B-Instruct", token=HF_TOKEN)

## 3. İkili (Pairwise) Karşılaştırmalar ve Kazanma Oranı Matrisi

> **Maliyet notu:** `N` test istemi ve `M` model için $N \times \binom{M}{2}$
> adet hakem çağrısı yapılır (burada $M=4$ model için $\binom{4}{2}=6$
> karşılaştırma/istem). Test istemi sayısını veya modelleri azaltarak API
> maliyetini kontrol edebilirsiniz.

In [ ]:
comparisons_df = ap.run_pairwise_comparisons(
    test_prompts=TEST_PROMPTS,
    responses_by_model=responses_by_model,
    judge_fn=judge_fn,
    seed=42,
)
comparisons_df.to_csv(os.path.join(QUAL_DIR, "pairwise_comparisons.csv"), index=False)
comparisons_df.head(10)

In [ ]:
win_rate_df = ap.compute_win_rate_matrix(comparisons_df, model_names=list(responses_by_model.keys()))
win_rate_df.to_csv(os.path.join(QUAL_DIR, "win_rate_matrix.csv"))
win_rate_df

In [ ]:
ap.plot_win_rate_heatmap(
    win_rate_df,
    title="İkili Kazanma Oranı (%) Matrisi — LLM-as-a-Judge (GPT-4o)",
    save_path=os.path.join(QUAL_DIR, "win_rate_heatmap.png"),
)

## 4. Gözlemler

- Satır modeli, sütun modeline karşı kazanma oranını (%) gösterir; eşitlikler
  0.5 kazanç olarak sayılmıştır.
- `Model_RLHF` ve `Model_DPO`'nun `Model_SFT`'ye karşı kazanma oranları,
  hizalamanın (alignment) insan tercihleriyle ne ölçüde örtüştüğünün nitel bir
  göstergesidir (Zheng vd., 2023).
- Nicel bulgularla (`05_Algorithm_Comparison_Quantitative.ipynb` — özellikle
  yanıt uzunluğu ve $D_{KL}$ sapması) birlikte yorumlandığında, yüksek
  kazanma oranına sahip ama aşırı uzun yanıtlar üreten bir model, ödül
  "hackleme" (reward hacking / length exploitation) şüphesi taşır.